# Homework 8 – Spark SQL Solution

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('hw8').getOrCreate()

## Part 1

In [2]:
missions = spark.read.csv('drone_missions.csv', header=True, inferSchema=True)
drones = spark.read.csv('drone_info.csv', header=True, inferSchema=True)

missions.show(10)
drones.show(10)

+---------+-------+----------+-----------+---------------+------------------+
|MissionID|DroneID|OperatorID|MissionType|DurationMinutes|BatteryUsedPercent|
+---------+-------+----------+-----------+---------------+------------------+
|      101|   DR-1|     OP-10|     Survey|             45|                40|
|      102|   DR-2|     OP-11|   Delivery|             30|                55|
|      103|   DR-3|     OP-12| Inspection|             60|                65|
|      104|   DR-1|     OP-10|     Survey|             50|                45|
|      105|   DR-4|     OP-13|   Delivery|             25|                50|
|      106|   DR-2|     OP-11| Inspection|             70|                70|
|      107|   DR-3|     OP-14|     Survey|             40|                35|
|      108|   DR-5|     OP-15|   Delivery|             35|                60|
+---------+-------+----------+-----------+---------------+------------------+

+-------+--------+----------------+--------+
|DroneID|   Model|

## Part 2

In [3]:
missions.write.mode('overwrite').saveAsTable('drone_missions')
drones.write.mode('overwrite').saveAsTable('drone_info')

## Part 3

In [4]:
spark.sql('''
CREATE TABLE drone_maintenance (
MaintenanceID INT,
DroneID STRING,
MaintenanceType STRING,
MaintenanceCost DOUBLE
)''')

DataFrame[]

In [5]:
spark.sql("""
INSERT INTO drone_maintenance VALUES
(1,'DR-1','Battery Replacement',120.0),
(2,'DR-2','Motor Repair',250.0),
(3,'DR-3','Propeller Replacement',75.0),
(4,'DR-1','Sensor Calibration',40.0),
(5,'DR-4','Battery Replacement',110.0)
""")

DataFrame[]

In [9]:
spark.sql('SELECT * FROM drone_maintenance').show()

+-------------+-------+--------------------+---------------+
|MaintenanceID|DroneID|     MaintenanceType|MaintenanceCost|
+-------------+-------+--------------------+---------------+
|            3|   DR-3|Propeller Replace...|           75.0|
|            4|   DR-1|  Sensor Calibration|           40.0|
|            5|   DR-4| Battery Replacement|          110.0|
|            1|   DR-1| Battery Replacement|          120.0|
|            2|   DR-2|        Motor Repair|          250.0|
+-------------+-------+--------------------+---------------+



## Part 4

In [14]:
joined = spark.sql('''
SELECT m.*, d.Model, d.MaxFlightMinutes, d.WeightKg,
BatteryUsedPercent / DurationMinutes AS EnergyUsage
FROM drone_missions m
JOIN drone_info d
ON m.DroneID = d.DroneID
''')

In [16]:
joined.show()

+---------+-------+----------+-----------+---------------+------------------+--------+----------------+--------+------------------+
|MissionID|DroneID|OperatorID|MissionType|DurationMinutes|BatteryUsedPercent|   Model|MaxFlightMinutes|WeightKg|       EnergyUsage|
+---------+-------+----------+-----------+---------------+------------------+--------+----------------+--------+------------------+
|      101|   DR-1|     OP-10|     Survey|             45|                40|Falcon-A|              60|     3.2|0.8888888888888888|
|      102|   DR-2|     OP-11|   Delivery|             30|                55|Falcon-B|              50|     3.5|1.8333333333333333|
|      103|   DR-3|     OP-12| Inspection|             60|                65|Falcon-A|              65|     3.2|1.0833333333333333|
|      104|   DR-1|     OP-10|     Survey|             50|                45|Falcon-A|              60|     3.2|               0.9|
|      105|   DR-4|     OP-13|   Delivery|             25|                50

In [15]:
joined.createOrReplaceTempView('mission_joined')

## Part 5

In [17]:
spark.sql('''
SELECT *,
CASE
WHEN DurationMinutes >= 60 THEN 'High'
WHEN DurationMinutes BETWEEN 40 AND 59 THEN 'Medium'
ELSE 'Low'
END AS MissionIntensity
FROM mission_joined
''').show()

+---------+-------+----------+-----------+---------------+------------------+--------+----------------+--------+------------------+----------------+
|MissionID|DroneID|OperatorID|MissionType|DurationMinutes|BatteryUsedPercent|   Model|MaxFlightMinutes|WeightKg|       EnergyUsage|MissionIntensity|
+---------+-------+----------+-----------+---------------+------------------+--------+----------------+--------+------------------+----------------+
|      101|   DR-1|     OP-10|     Survey|             45|                40|Falcon-A|              60|     3.2|0.8888888888888888|          Medium|
|      102|   DR-2|     OP-11|   Delivery|             30|                55|Falcon-B|              50|     3.5|1.8333333333333333|             Low|
|      103|   DR-3|     OP-12| Inspection|             60|                65|Falcon-A|              65|     3.2|1.0833333333333333|            High|
|      104|   DR-1|     OP-10|     Survey|             50|                45|Falcon-A|              60|   

## Part 6 UDF

In [18]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def stress(duration, maxflight, weight):
    return (duration / maxflight) * weight

# stress_udf = udf(stress, DoubleType())

spark.udf.register("estimate_stress_score", stress, DoubleType())

<function __main__.stress(duration, maxflight, weight)>

In [23]:
another_view = spark.sql("""
SELECT *,
       estimate_stress_score(DurationMinutes, MaxFlightMinutes, WeightKg) AS StressScore
FROM mission_joined
""")

In [26]:
# create a view
another_view.createOrReplaceTempView('mission_scored')

In [27]:
another_view.show()

+---------+-------+----------+-----------+---------------+------------------+--------+----------------+--------+------------------+------------------+
|MissionID|DroneID|OperatorID|MissionType|DurationMinutes|BatteryUsedPercent|   Model|MaxFlightMinutes|WeightKg|       EnergyUsage|       StressScore|
+---------+-------+----------+-----------+---------------+------------------+--------+----------------+--------+------------------+------------------+
|      101|   DR-1|     OP-10|     Survey|             45|                40|Falcon-A|              60|     3.2|0.8888888888888888|2.4000000000000004|
|      102|   DR-2|     OP-11|   Delivery|             30|                55|Falcon-B|              50|     3.5|1.8333333333333333|               2.1|
|      103|   DR-3|     OP-12| Inspection|             60|                65|Falcon-A|              65|     3.2|1.0833333333333333| 2.953846153846154|
|      104|   DR-1|     OP-10|     Survey|             50|                45|Falcon-A|        

## Part 7

In [33]:
spark.sql('''
CREATE OR REPLACE TEMP VIEW drone_operations_summary AS
SELECT
d.Model,
m.MissionType,
COUNT(*) AS TotalMissions,
AVG(DurationMinutes) AS AverageDuration,
AVG(StressScore) AS AverageStressScore,
SUM(MaintenanceCost) AS TotalMaintenanceCost
FROM mission_scored m
JOIN drone_info d ON m.DroneID = d.DroneID
LEFT JOIN drone_maintenance dm ON m.DroneID = dm.DroneID
GROUP BY d.Model, m.MissionType
''')

DataFrame[]

In [35]:
spark.sql('SELECT * FROM drone_operations_summary').show()

+--------+-----------+-------------+---------------+------------------+--------------------+
|   Model|MissionType|TotalMissions|AverageDuration|AverageStressScore|TotalMaintenanceCost|
+--------+-----------+-------------+---------------+------------------+--------------------+
|Falcon-C|   Delivery|            1|           35.0| 2.418181818181818|                NULL|
|Falcon-A| Inspection|            1|           60.0| 2.953846153846154|                75.0|
|Falcon-B| Inspection|            1|           70.0|4.8999999999999995|               250.0|
|  Hawk-X|   Delivery|            1|           25.0|           1.59375|               110.0|
|Falcon-B|   Delivery|            1|           30.0|               2.1|               250.0|
|Falcon-A|     Survey|            5|           46.0| 2.420512820512821|               395.0|
+--------+-----------+-------------+---------------+------------------+--------------------+



## Part 8

In [34]:
spark.sql('''
SELECT *
FROM drone_operations_summary
ORDER BY AverageStressScore DESC
''').show()

+--------+-----------+-------------+---------------+------------------+--------------------+
|   Model|MissionType|TotalMissions|AverageDuration|AverageStressScore|TotalMaintenanceCost|
+--------+-----------+-------------+---------------+------------------+--------------------+
|Falcon-B| Inspection|            1|           70.0|4.8999999999999995|               250.0|
|Falcon-A| Inspection|            1|           60.0| 2.953846153846154|                75.0|
|Falcon-A|     Survey|            5|           46.0| 2.420512820512821|               395.0|
|Falcon-C|   Delivery|            1|           35.0| 2.418181818181818|                NULL|
|Falcon-B|   Delivery|            1|           30.0|               2.1|               250.0|
|  Hawk-X|   Delivery|            1|           25.0|           1.59375|               110.0|
+--------+-----------+-------------+---------------+------------------+--------------------+

